<a href="https://colab.research.google.com/github/Shifali1094/stance-detection/blob/main/notebooks/bertweet_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# =========================
# 1. Install libraries
# =========================
!pip install -q transformers datasets evaluate accelerate emoji vncorenlp

In [3]:
# =========================
# 2. Imports
# =========================
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [4]:
# =========================
# 3. Load dataset
# CHANGE THESE
# =========================

DATA_PATH = "../data/cleaned_dataset.csv"   # change this path
TEXT_COL = "text"                          # change if needed
LABEL_COL = "label"                        # change if needed

df = pd.read_csv(DATA_PATH)

df = df[[TEXT_COL, LABEL_COL]].dropna()
df[TEXT_COL] = df[TEXT_COL].astype(str)

df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/cleaned_dataset.csv'

In [ ]:
# =========================
# 4. Label encoding
# =========================

labels = sorted(df[LABEL_COL].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

df["labels"] = df[LABEL_COL].map(label2id)

print("Labels:", label2id)
print(df["labels"].value_counts())

In [ ]:
# =========================
# 5. Train/validation/test split
# =========================

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["labels"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["labels"]
)

print(len(train_df), len(val_df), len(test_df))

In [ ]:
# =========================
# 6. Convert to Hugging Face Dataset
# =========================

train_dataset = Dataset.from_pandas(train_df[[TEXT_COL, "labels"]])
val_dataset = Dataset.from_pandas(val_df[[TEXT_COL, "labels"]])
test_dataset = Dataset.from_pandas(test_df[[TEXT_COL, "labels"]])

In [ ]:
# =========================
# 7. Load BERTweet tokenizer
# =========================

MODEL_NAME = "vinai/bertweet-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
    normalization=True
)

def tokenize_function(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

In [ ]:
# =========================
# 8. Load BERTweet model
# =========================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# =========================
# 9. Metrics function
# =========================

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    acc = accuracy_score(y_true, y_pred)

    return {
        "accuracy": acc,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }

In [ ]:
# =========================
# 10. Training settings
# =========================

training_args = TrainingArguments(
    output_dir="./bertweet_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./bertweet_logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    seed=42
)

In [ ]:
# =========================
# 11. Train model
# =========================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
# =========================
# 12. Evaluate on test set
# =========================

test_results = trainer.evaluate(test_dataset)
test_results

In [ ]:
# =========================
# 13. Predictions
# =========================

predictions = trainer.predict(test_dataset)

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print(classification_report(
    y_true,
    y_pred,
    target_names=[str(id2label[i]) for i in range(len(labels))],
    zero_division=0
))

In [ ]:
# =========================
# 14. Confusion matrix
# =========================

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[str(id2label[i]) for i in range(len(labels))]
)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, xticks_rotation=45)
plt.title("BERTweet Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 15. Save predictions
# =========================

test_output = test_df.copy()
test_output["predicted_label_id"] = y_pred
test_output["predicted_label"] = [id2label[i] for i in y_pred]
test_output["true_label"] = [id2label[i] for i in y_true]

test_output.to_csv("bertweet_test_predictions.csv", index=False)

test_output.head()

In [ ]:
# =========================
# 16. Save final model
# =========================

trainer.save_model("./final_bertweet_model")
tokenizer.save_pretrained("./final_bertweet_model")